# Practice: Merging and Datatypes

In this exercise, you will practice changing datatypes, merging two datasets together, and finally visualizing the result.

## 0. Introduction: pd.concat and pd.merge with Real Data

Before we dive into the full analysis, let's explore `pd.concat` and `pd.merge` using real data from the World Bank.

In [ ]:
import pandas as pd
import requests
import io

def get_wb_data(indicator, country, start_year, end_year):
    url = f'https://api.worldbank.org/v2/country/{country}/indicator/{indicator}?date={start_year}:{end_year}&format=json&per_page=1000'
    response = requests.get(url)
    data = response.json()[1]
    df = pd.json_normalize(data)
    return df[['countryiso3code', 'date', 'value']]

# Pre-loading some data for examples
usa_pop = get_wb_data('SP.POP.TOTL', 'USA', 2020, 2022)
can_pop = get_wb_data('SP.POP.TOTL', 'CAN', 2020, 2022)
usa_gdp = get_wb_data('NY.GDP.PCAP.CD', 'USA', 2020, 2022)

### 0.1 pd.concat (Stacking Data)
We can use `pd.concat` to combine the USA population data and Canada population data into one table.

In [ ]:
print("USA Population Subset:")
print(usa_pop)
print("\nCAN Population Subset:")
print(can_pop)

# Combine them vertically
combined_pop = pd.concat([usa_pop, can_pop], axis=0)
print("\nCombined Population (Stacked):")
print(combined_pop)

### 0.2 pd.merge (Joining Data)
We can use `pd.merge` to join the USA Population data with the USA GDP data based on the date.

In [ ]:
# Rename columns to avoid confusion
usa_pop_renamed = usa_pop.rename(columns={'value': 'population'})
usa_gdp_renamed = usa_gdp.rename(columns={'value': 'gdp_per_capita'})

merged_usa = pd.merge(usa_pop_renamed, usa_gdp_renamed, on=['countryiso3code', 'date'])
print("Merged USA Data (Population + GDP):")
print(merged_usa)

## 1. Project: Advanced Merging with Functions
Now let's apply these concepts in a structured way using simple functions. We will download macroeconomic and OECD data. Run the cell below.

In [ ]:
def download_macro_data(url):
    print(f"Fetching macroeconomic data from: {url}")
    df = pd.read_stata(url)
    return df

def download_oecd_data(url):
    print(f"Fetching OECD data from: {url}")
    response = requests.get(url)
    response.raise_for_status()
    data = io.StringIO(response.text)
    df = pd.read_csv(data)
    return df

url_macro = 'https://github.com/KMueller-Lab/Global-Macro-Database/raw/refs/heads/main/data/final/chainlinked_infl.dta'
url_oecd = "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_PDB@DF_PDB_ULC_Q,1.0/.Q.......?startPeriod=1990-Q4&format=csv"

df_macro_raw = download_macro_data(url_macro)
df_oecd_raw = download_oecd_data(url_oecd)

## 2. Filtering Data
Filter the data for New Zealand (NZL) and rename columns. Try to fill in the missing code!

In [ ]:
def filter_rename_macro_nz(df):
    df_nz = df.query("ISO3 == 'NZL'")[['ISO3', 'year', 'OECD_KEI_infl', 'BIS_infl']].dropna()
    df_nz = df_nz.rename({"ISO3":'country', "year":'date'}, axis=1)
    return df_nz

def filter_rename_oecd_nz(df):
    # YOUR CODE HERE to filter for NZL, MEASURE=='ULCE', UNIT_MEASURE=='PA'
    cols = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE', 'MEASURE', 'UNIT_MEASURE']
    df_nz = df[cols].query("REF_AREA == 'NZL' & MEASURE=='ULCE' & UNIT_MEASURE == 'PA'")
    df_nz = df_nz.rename({"REF_AREA":'country', "TIME_PERIOD":'date', 'OBS_VALUE':'ULCE'}, axis=1).drop(["MEASURE", "UNIT_MEASURE"], axis=1)
    return df_nz

df_macro_nz = filter_rename_macro_nz(df_macro_raw.copy())
df_oecd_nz = filter_rename_oecd_nz(df_oecd_raw.copy())

## 3. Convert Datetimes
Convert your data columns to datetime types.

In [ ]:
def convert_datetime(df_oecd, df_macro):
    df_oecd['date'] = pd.PeriodIndex(df_oecd['date'], freq='Q').to_timestamp().date
    df_macro['date'] = pd.to_datetime(df_macro['date'], format='%Y').dt.date
    return df_oecd, df_macro

df_oecd_nz_dt, df_macro_nz_dt = convert_datetime(df_oecd_nz.copy(), df_macro_nz.copy())

## 4. Set Index
Set both 'country' and 'date' as index for merging properly.

In [ ]:
def set_index(df_oecd, df_macro):
    # YOUR CODE HERE
    df_macro = df_macro.set_index(['country', 'date'])
    df_oecd = df_oecd.set_index(['country', 'date'])
    return df_oecd, df_macro

df_oecd_indexed, df_macro_indexed = set_index(df_oecd_nz_dt.copy(), df_macro_nz_dt.copy())

## 5. Merging
Merge the macroeconomic and OECD datasets together on their indices.

In [ ]:
def merge_data(df_macro, df_oecd):
    # YOUR CODE HERE
    df_merge = pd.merge(
        df_macro,
        df_oecd,
        right_index=True,
        left_index=True,
        how='inner'
    )
    return df_merge

df_merged = merge_data(df_macro_indexed.copy(), df_oecd_indexed.copy())
df_merged.head()

## 6. Visualizing the Merge
Let's visualize the relationship between OECD inflation and BIS inflation.

In [ ]:
# Create scatter plot
plt.figure(figsize=(8, 5))
plt.scatter(df_merged['OECD_KEI_infl'], df_merged['BIS_infl'])
plt.xlabel('OECD KEI Inflation')
plt.ylabel('BIS Inflation')
plt.title('OECD vs BIS Inflation (NZL)')
plt.grid(True)
plt.show()